In [ ]:
import sys
import os
import json
import re
import pandas as pd
from datetime import datetime, timezone

# 1. DB 연결 함수
def get_connection():
    from idcube_hive_connector import connector
    return connector.connect_idcube_athena()

# 2. WKT 좌표 파싱 함수
def parse_wkt_polygon(wkt: str):
    if not wkt:
        return None
    try:
        from shapely import wkt as shapely_wkt
        geom = shapely_wkt.loads(wkt)
        if geom.geom_type == 'Polygon':
            return [[x, y] for x, y in geom.exterior.coords]
        if geom.geom_type == 'MultiPolygon':
            first = list(geom.geoms)[0]
            return [[x, y] for x, y in first.exterior.coords]
        return None
    except Exception:
        pass
    m = re.search(r'\(\(([^()]+)\)\)', wkt)
    if not m:
        return None
    points = []
    for pair in m.group(1).split(','):
        parts = pair.strip().split()
        if len(parts) >= 2:
            try: points.append([float(parts[0]), float(parts[1])])
            except ValueError: continue
    return points if len(points) >= 3 else None

# 3. Athena 단지 폴리곤 조회
def find_complex_polygon(conn, lat: float, lng: float):
    query = f"""
        SELECT bld_cplx_inf_id, cplx_type_div_id, cplx_lcl_nm, cplx_mcl_nm, cplx_scl_nm,
               ST_AsText(ST_GeomFromBinary(pygn_geo)) AS wkt
        FROM o_moira.ap_bld_cplx_inf
        WHERE cplx_mcl_nm = '아파트'
          AND ST_Contains(ST_GeomFromBinary(pygn_geo), ST_Point({lng:.8f}, {lat:.8f}))
        LIMIT 1
    """
    df = pd.read_sql_query(query, conn)
    if df.empty: return None
    row = df.iloc[0]
    polygon = parse_wkt_polygon(row['wkt'])
    if not polygon: return None
    return {
        'bld_cplx_inf_id': int(row['bld_cplx_inf_id']) if row['bld_cplx_inf_id'] is not None else None,
        'cplx_scl_nm': row.get('cplx_scl_nm'),
        'polygon': polygon,
    }

# 4. Athena 단지 내 건물 조회 (단지 ID 직접 조인 - 버퍼 없이 순수 단지 내부 건물만 정확히 조회)
def find_buildings_in_complex(conn, complex_id: int):
    query = f"""
        SELECT b.ciss_bld_cd, b.bld_nm, b.bld_lcl_nm, b.bld_scl_nm,
               b.grud_flor_cnt, b.bsmt_flor_cnt, b.bld_hght,
               b.bld_in_lng, b.bld_in_lat,
               ST_AsText(ST_GeomFromBinary(b.geo)) AS wkt
        FROM o_moira.ap_bld_cplx_inf c
        JOIN o_moira.ac_bld_bas b
          ON ST_Intersects(ST_GeomFromBinary(c.pygn_geo), ST_Point(b.bld_in_lng, b.bld_in_lat))
        WHERE c.bld_cplx_inf_id = '{complex_id}'
    """
    return pd.read_sql_query(query, conn)

# 5. 폴백용 ~400m 반경 건물 조회
def find_buildings_bbox_fallback(conn, lat: float, lng: float, buffer_deg: float = 0.0036, bld_usg_filter: str = None):
    where_filter = f"AND bld_lcl_nm = '{bld_usg_filter}'" if bld_usg_filter else ""
    query = f"""
        SELECT ciss_bld_cd, bld_nm, bld_lcl_nm, bld_scl_nm,
               grud_flor_cnt, bsmt_flor_cnt, bld_hght,
               bld_in_lng, bld_in_lat,
               ST_AsText(ST_GeomFromBinary(geo)) AS wkt
        FROM o_moira.ac_bld_bas
        WHERE bld_in_lng BETWEEN {lng - buffer_deg:.8f} AND {lng + buffer_deg:.8f}
          AND bld_in_lat BETWEEN {lat - buffer_deg:.8f} AND {lat + buffer_deg:.8f}
          {where_filter}
    """
    return pd.read_sql_query(query, conn)

def buildings_df_to_list(df):
    buildings = []
    for _, row in df.iterrows():
        buildings.append({
            'ciss_bld_cd': row.get('ciss_bld_cd'),
            'bld_nm': row.get('bld_nm'),
            'floors_above': None if pd.isna(row.get('grud_flor_cnt')) else int(row.get('grud_flor_cnt')),
            'floors_below': None if pd.isna(row.get('bsmt_flor_cnt')) else int(row.get('bsmt_flor_cnt')),
            'height': None if pd.isna(row.get('bld_hght')) else float(row.get('bld_hght')),
            'center': [row.get('bld_in_lng'), row.get('bld_in_lat')],
            'polygon': parse_wkt_polygon(row.get('wkt')),
        })
    return buildings

# 6. 대상 1건 조회 함수 (단지 발견 시 순수 단지 조인, 미발견 시 400m 가상 단지 폴리곤)
def query_one_target(conn, rapa_key: str, lat: float, lng: float, fallback_400m: float = 0.0036, bld_usg_filter: str = None):
    result = {
        'rapaKey': rapa_key,
        'queriedAt': datetime.now(timezone.utc).isoformat(),
        'complex': None,
        'buildingSource': None,
        'buildings': [],
    }
    try:
        complex_info = find_complex_polygon(conn, lat, lng)
    except Exception as e:
        complex_info = None

    if complex_info and complex_info.get('bld_cplx_inf_id'):
        # [케이스 1] 단지 발견 → 단지 ID로 단지 내부 건물만 정확히 조인 조회
        result['complex'] = complex_info
        complex_id = complex_info['bld_cplx_inf_id']
        try:
            bdf = find_buildings_in_complex(conn, complex_id)
            result['buildingSource'] = 'complex_polygon'
        except Exception as e:
            bdf = find_buildings_bbox_fallback(conn, lat, lng, fallback_400m, bld_usg_filter)
            result['buildingSource'] = 'virtual_400m_bbox'
    else:
        # [케이스 2] 단지 미발견 → 400m 사각형 가상 단지 폴리곤 생성 및 해당 영역 건물 조회
        d = fallback_400m
        virtual_polygon = [
            [lng - d, lat - d],
            [lng + d, lat - d],
            [lng + d, lat + d],
            [lng - d, lat + d],
            [lng - d, lat - d],
        ]
        result['complex'] = {
            'bld_cplx_inf_id': None,
            'cplx_scl_nm': '가상영역(400m)',
            'polygon': virtual_polygon,
        }
        bdf = find_buildings_bbox_fallback(conn, lat, lng, fallback_400m, bld_usg_filter)
        result['buildingSource'] = 'virtual_400m_bbox'

    result['buildings'] = buildings_df_to_list(bdf)
    return result

# 7. 일괄 배치 실행 함수
def cmd_batch(args):
    import csv as csv_mod
    conn = get_connection()
    with open(args.csv, encoding='utf-8-sig') as f:
        rows = list(csv_mod.DictReader(f))

    if getattr(args, 'rapa_key', None):
        rows = [r for r in rows if r.get('RAPA식별코드') in args.rapa_key]
    if getattr(args, 'limit', None):
        rows = rows[:args.limit]

    print(f"총 {len(rows)}건 조회 시작...")
    targets = {}
    failed = []
    for idx, row in enumerate(rows):
        rapa_key = row.get('RAPA식별코드')
        try:
            lat = float(row.get('위도'))
            lng = float(row.get('경도'))
            fallback_400m = getattr(args, 'fallback_400m', 0.0036)
            bld_usg_filter = getattr(args, 'bld_usg_filter', None)
            
            targets[rapa_key] = query_one_target(
                conn, rapa_key, lat, lng,
                fallback_400m=fallback_400m,
                bld_usg_filter=bld_usg_filter
            )
            b_cnt = len(targets[rapa_key]['buildings'])
            src = targets[rapa_key]['buildingSource']
            print(f"[{idx+1}/{len(rows)}] {rapa_key} 완료 ({src}, 건물 {b_cnt}개)")
        except Exception as e:
            print(f"[실패] {rapa_key}: {e}")
            failed.append(rapa_key)

    output = {
        'generatedAt': datetime.now(timezone.utc).isoformat(),
        'sourceCsv': args.csv,
        'failed': failed,
        'targets': targets,
    }
    if getattr(args, 'out', None):
        with open(args.out, 'w', encoding='utf-8') as f:
            json.dump(output, f, ensure_ascii=False, indent=2)
        print(f"🎉 배치 완료! 저장 파일: {args.out}")
    return output


In [ ]:
# 옵션 정의 (Class 사용)
class Args:
    csv = 'apt_list.csv'                  # 입력 CSV 경로
    out = 'public/complex_polygons.json' # 저장할 JSON 경로
    rapa_key = None                       # 특정 단지만 하려면 ['m-RAPA-1811-6729']
    limit = None                          # 테스트로 5개만 먼저 해볼 때 (전체는 None)
    fallback_400m = 0.0036                # 단지 미발견 시 가상 단지 반경 (400m)
    bld_usg_filter = None                 # 건물 용도 필터 (None = 전체 용도)

# 배치 실행
cmd_batch(Args())


In [ ]:
# 저장된 JSON 파일 열어서 확인
with open('public/complex_polygons.json', encoding='utf-8') as f:
    data = json.load(f)

print("총 성공 단지 수:", len(data['targets']))

# 첫 번째 단지의 건물 목록 확인
first_key = list(data['targets'].keys())[0]
first_item = data['targets'][first_key]
print(f"단지: {first_key}")
print(f"단지 정보: {first_item['complex']}")
print(f"건물 목록:")
display(pd.DataFrame(first_item['buildings']))
